<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/mtbench_typo_matching_stratified_typo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import random
import os
import pandas as pd
from google.colab import files


# =========================================================
# 1. Settings
# =========================================================

RANDOM_SEED = 42

# 抽取 5 个共同匹配成功的原始 prompt
# 每个原始 prompt 保留 3 个 typo-rate 版本
# 最终：5 × 3 = 15 行
SAMPLE_SIZE = 5

BENCHMARK_NAME = "MTBench"

TYPO_RATES = ["0.1", "0.4", "0.7"]


# =========================================================
# 2. File paths
# =========================================================

FILE_PATHS = {
    "0.1": "/content/raw_0.1.jsonl",
    "0.4": "/content/raw_0.4.jsonl",
    "0.7": "/content/raw_0.7.jsonl"
}


print("Checking input files:\n")

for typo_rate, file_path in FILE_PATHS.items():

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            f"Please upload raw_{typo_rate}.jsonl to Colab."
        )

    print(f"Rate {typo_rate}: {file_path}")


# =========================================================
# 3. Helper functions
# =========================================================

def read_jsonl(file_path):
    """
    Read a JSONL file and preserve its original row number.
    """

    records = []

    with open(file_path, "r", encoding="utf-8-sig") as file:

        for line_number, line in enumerate(file, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number}\n"
                    f"File: {file_path}\n"
                    f"Error: {error}"
                )

            record["_source_row"] = line_number
            records.append(record)

    return records


def normalize_text(value):
    """
    Normalize text only for matching.

    It removes unnecessary differences in spaces
    and line breaks without changing the text shown
    in the final CSV.
    """

    if value is None:
        return ""

    if isinstance(value, (dict, list)):
        value = json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True
        )

    return " ".join(str(value).split())


def convert_to_cell(value):
    """
    Convert dictionaries and lists into readable CSV text.
    """

    if value is None:
        return ""

    if isinstance(value, (dict, list)):
        return json.dumps(
            value,
            ensure_ascii=False
        )

    return str(value)


def get_first_available(record, possible_fields):
    """
    Return the first non-empty field found in the record.
    """

    for field_name in possible_fields:

        value = record.get(field_name)

        if value is None:
            continue

        if isinstance(value, (list, dict)):
            if len(value) > 0:
                return value

        elif str(value).strip() != "":
            return value

    return ""


def get_original_prompt(record):
    """
    Extract the original MT-Bench prompt used for matching.
    """

    return get_first_available(
        record,
        [
            "original_text_backup",
            "original_prompt",
            "original_question",
            "original_instruction",
            "original_text",
            "prompt_original",
            "original_turns"
        ]
    )


def get_modified_prompt(record):
    """
    Extract the typo-transformed MT-Bench prompt.

    MT-Bench may store prompts in 'turns', so that
    field is checked first.
    """

    return get_first_available(
        record,
        [
            "turns",
            "question",
            "prompt",
            "modified_prompt",
            "instruction",
            "text",
            "input"
        ]
    )


def get_gold_answer(record):
    """
    Extract a reference or gold answer if present.

    MT-Bench does not always have a conventional gold answer,
    so empty values may be normal.
    """

    return get_first_available(
        record,
        [
            "answer",
            "reference_answer",
            "gold_answer",
            "output",
            "response",
            "reference",
            "model_answer"
        ]
    )


def get_question_id(record):
    """
    Extract the MT-Bench question identifier if present.
    """

    return get_first_available(
        record,
        [
            "question_id",
            "questionId",
            "id",
            "qid"
        ]
    )


def get_category(record):
    """
    Extract the MT-Bench category if present.
    """

    return get_first_available(
        record,
        [
            "category",
            "question_category",
            "task_category",
            "domain"
        ]
    )


def get_reference_answer(record):
    """
    Extract an additional MT-Bench reference answer,
    if the source file stores it separately.
    """

    return get_first_available(
        record,
        [
            "reference_answer",
            "reference",
            "ref_answer",
            "judge_reference"
        ]
    )


# =========================================================
# 4. Read all three JSONL files
# =========================================================

datasets = {}

print("\nLoading files:\n")

for typo_rate, file_path in FILE_PATHS.items():

    datasets[typo_rate] = read_jsonl(file_path)

    print(
        f"Loaded {len(datasets[typo_rate])} records "
        f"for typo rate {typo_rate}"
    )


# Show the available fields
print("\nFields found in the first record of each file:")

for typo_rate in TYPO_RATES:

    if datasets[typo_rate]:

        print(
            f"\nRate {typo_rate}: "
            f"{list(datasets[typo_rate][0].keys())}"
        )


# =========================================================
# 5. Build matching maps
# =========================================================

record_maps = {}

print("\nBuilding matching maps:")

for typo_rate, records in datasets.items():

    current_map = {}

    missing_original_count = 0
    duplicate_count = 0

    for record in records:

        original_prompt = get_original_prompt(record)

        normalized_original = normalize_text(
            original_prompt
        )

        if not normalized_original:
            missing_original_count += 1
            continue

        if normalized_original in current_map:
            duplicate_count += 1
            continue

        current_map[normalized_original] = record

    record_maps[typo_rate] = current_map

    print(
        f"\nRate {typo_rate}: "
        f"{len(current_map)} usable original prompts"
    )

    if missing_original_count > 0:
        print(
            f"Warning: {missing_original_count} records "
            f"had no usable original prompt."
        )

    if duplicate_count > 0:
        print(
            f"Warning: {duplicate_count} duplicate original prompts "
            f"were found. The first record was kept."
        )


# =========================================================
# 6. Find prompts shared by all three typo-rate files
# =========================================================

common_original_keys = set(
    record_maps["0.1"].keys()
)

for typo_rate in ["0.4", "0.7"]:

    common_original_keys &= set(
        record_maps[typo_rate].keys()
    )


# Sort before sampling for full reproducibility
common_original_keys = sorted(
    common_original_keys
)


print(
    f"\nCommon matched original prompts: "
    f"{len(common_original_keys)}"
)


if len(common_original_keys) < SAMPLE_SIZE:

    raise ValueError(
        f"Only {len(common_original_keys)} matched prompts "
        f"were found across all three files.\n"
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


# =========================================================
# 7. Randomly sample 5 matched original prompts
# =========================================================

random.seed(RANDOM_SEED)

selected_original_keys = random.sample(
    common_original_keys,
    SAMPLE_SIZE
)


print(f"\nRandom seed: {RANDOM_SEED}")

print(
    f"Selected matched prompts: "
    f"{len(selected_original_keys)}"
)


# =========================================================
# 8. Create matched stratified output
# =========================================================

output_rows = []


for prompt_number, original_key in enumerate(
    selected_original_keys,
    start=1
):

    base_id = (
        f"{BENCHMARK_NAME}_{prompt_number:03d}"
    )

    # Use rate 0.1 as the reference source
    reference_record = record_maps["0.1"][
        original_key
    ]

    original_prompt = get_original_prompt(
        reference_record
    )

    question_id = get_question_id(
        reference_record
    )

    category = get_category(
        reference_record
    )

    for typo_rate in TYPO_RATES:

        record = record_maps[typo_rate][
            original_key
        ]

        modified_prompt = get_modified_prompt(
            record
        )

        gold_answer = get_gold_answer(
            record
        )

        reference_answer = get_reference_answer(
            record
        )

        output_rows.append({

            "Base_ID": base_id,

            "Sample_ID": (
                f"{base_id}_rate_{typo_rate}"
            ),

            "Benchmark": BENCHMARK_NAME,

            "Typo_Rate": typo_rate,

            "Source_File": os.path.basename(
                FILE_PATHS[typo_rate]
            ),

            "Source_Row": record.get(
                "_source_row",
                ""
            ),

            "Question_ID": convert_to_cell(
                question_id
            ),

            "Category": convert_to_cell(
                category
            ),

            "Original_Prompt": convert_to_cell(
                original_prompt
            ),

            "Modified_Prompt": convert_to_cell(
                modified_prompt
            ),

            "Gold_Answer": convert_to_cell(
                gold_answer
            ),

            "Reference_Answer": convert_to_cell(
                reference_answer
            ),

            # Reviewer 1
            "R1_Meaning": "",
            "R1_Key_Info": "",
            "R1_Answer_Preserved": "",
            "R1_Realism": "",
            "R1_Readability": "",
            "R1_Comments": "",

            # Reviewer 2
            "R2_Meaning": "",
            "R2_Key_Info": "",
            "R2_Answer_Preserved": "",
            "R2_Realism": "",
            "R2_Readability": "",
            "R2_Comments": "",

            # Final adjudication
            "Final_Meaning": "",
            "Final_Key_Info": "",
            "Final_Answer_Preserved": "",
            "Final_Realism": "",
            "Final_Readability": "",
            "Final_Comments": ""
        })


# =========================================================
# 9. Convert to DataFrame
# =========================================================

sample_df = pd.DataFrame(
    output_rows
)


print("\nSampling completed.")

print(
    f"Selected original prompts: "
    f"{SAMPLE_SIZE}"
)

print(
    f"Number of typo-rate strata: "
    f"{len(TYPO_RATES)}"
)

print(
    f"Total output rows: "
    f"{len(sample_df)}"
)


display(sample_df)


# =========================================================
# 10. Validate important columns
# =========================================================

empty_original = (
    sample_df["Original_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_modified = (
    sample_df["Modified_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_gold_answer = (
    sample_df["Gold_Answer"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_question_id = (
    sample_df["Question_ID"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_category = (
    sample_df["Category"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)


print("\nColumn check:")

print(
    f"Empty Original_Prompt rows: "
    f"{empty_original}"
)

print(
    f"Empty Modified_Prompt rows: "
    f"{empty_modified}"
)

print(
    f"Empty Gold_Answer rows: "
    f"{empty_gold_answer}"
)

print(
    f"Empty Question_ID rows: "
    f"{empty_question_id}"
)

print(
    f"Empty Category rows: "
    f"{empty_category}"
)


# =========================================================
# 11. Check stratification and matching
# =========================================================

print("\nRows per typo-rate stratum:")

print(
    sample_df["Typo_Rate"]
    .value_counts()
    .sort_index()
)


# Every Base_ID should have exactly three rows
base_id_counts = (
    sample_df
    .groupby("Base_ID")
    .size()
)

incorrect_base_ids = base_id_counts[
    base_id_counts != len(TYPO_RATES)
]


if len(incorrect_base_ids) == 0:

    print(
        "\nMatched sampling check passed: "
        "every Base_ID has 3 typo-rate versions."
    )

else:

    print(
        "\nWarning: Some Base_ID values do not have "
        "exactly 3 typo-rate versions:"
    )

    print(incorrect_base_ids)


# Check whether each Base_ID has all three rate labels
expected_rates = set(TYPO_RATES)

rate_check = (
    sample_df
    .groupby("Base_ID")["Typo_Rate"]
    .apply(lambda values: set(values.astype(str)))
)

incorrect_rate_sets = rate_check[
    rate_check.apply(
        lambda values: values != expected_rates
    )
]


if len(incorrect_rate_sets) == 0:

    print(
        "Rate-label check passed: every Base_ID contains "
        "0.1, 0.4, and 0.7."
    )

else:

    print(
        "\nWarning: Some Base_ID values are missing "
        "one or more typo-rate labels:"
    )

    print(incorrect_rate_sets)


# =========================================================
# 12. Notes about optional MT-Bench fields
# =========================================================

if empty_original > 0:

    print(
        "\nWarning: Some Original_Prompt values are empty."
    )


if empty_modified > 0:

    print(
        "\nWarning: Some Modified_Prompt values are empty.\n"
        "Check the field names printed near the beginning "
        "of the output."
    )


if empty_gold_answer > 0:

    print(
        "\nNote: Empty Gold_Answer values may be normal "
        "for MT-Bench because it is commonly evaluated "
        "using an LLM judge rather than one fixed answer."
    )


if empty_question_id > 0:

    print(
        "\nNote: Some Question_ID values are empty. "
        "This does not affect prompt matching."
    )


if empty_category > 0:

    print(
        "\nNote: Some Category values are empty. "
        "This does not affect prompt matching."
    )


# =========================================================
# 13. Save and download CSV
# =========================================================

output_file = (
    "/content/"
    "MTBench_matched_stratified_sample_"
    "n5_seed42.csv"
)


sample_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"\nCSV saved successfully: "
    f"{output_file}"
)


files.download(output_file)

Checking input files:

Rate 0.1: /content/raw_0.1.jsonl
Rate 0.4: /content/raw_0.4.jsonl
Rate 0.7: /content/raw_0.7.jsonl

Loading files:

Loaded 80 records for typo rate 0.1
Loaded 80 records for typo rate 0.4
Loaded 80 records for typo rate 0.7

Fields found in the first record of each file:

Rate 0.1: ['question_id', 'category', 'turns', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Rate 0.4: ['question_id', 'category', 'turns', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Rate 0.7: ['question_id', 'category', 'turns', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Building matching maps:

Rate 0.1: 80 usable original prompts

Rate 0.4: 80 usable original prompts

Rate 0.7: 80 usable original prompts

Common matched original prompts: 80

Random seed: 42
Selected matched prompts: 5

Sampling completed.
Selected original prompts: 5
Number of typo-rate strata: 3
Total output r

,Base_ID,Sample_ID,Benchmark,Typo_Rate,Source_File,Source_Row,Question_ID,Category,Original_Prompt,Modified_Prompt,...,R2_Answer_Preserved,R2_Realism,R2_Readability,R2_Comments,Final_Meaning,Final_Key_Info,Final_Answer_Preserved,Final_Realism,Final_Readability,Final_Comments
0,MTBench_001,MTBench_001_rate_0.1,MTBench,0.1,raw_0.1.jsonl,65,145,stem,"[""Describe the process and write out the balan...","[""Describe the process and write out the balan...",...,,,,,,,,,,
1,MTBench_001,MTBench_001_rate_0.4,MTBench,0.4,raw_0.4.jsonl,65,145,stem,"[""Describe the process and write out the balan...","[""Descrbie the process and write out the balan...",...,,,,,,,,,,
2,MTBench_001,MTBench_001_rate_0.7,MTBench,0.7,raw_0.7.jsonl,65,145,stem,"[""Describe the process and write out the balan...","[""Descrive th proces and wrtiw out thed balanc...",...,,,,,,,,,,
3,MTBench_002,MTBench_002_rate_0.1,MTBench,0.1,raw_0.1.jsonl,17,97,roleplay,"[""Act as a math teacher. I will provide some m...","[""Act as a mat teacher. I will provide some ma...",...,,,,,,,,,,
4,MTBench_002,MTBench_002_rate_0.4,MTBench,0.4,raw_0.4.jsonl,17,97,roleplay,"[""Act as a math teacher. I will provide some m...","[""Acy aa a math teacher. I wil provide somne m...",...,,,,,,,,,,
5,MTBench_002,MTBench_002_rate_0.7,MTBench,0.7,raw_0.7.jsonl,17,97,roleplay,"[""Act as a math teacher. I will provide some m...","[""Act as a mat teacehr. I wikk provide sme mat...",...,,,,,,,,,,
6,MTBench_003,MTBench_003_rate_0.1,MTBench,0.1,raw_0.1.jsonl,55,135,extraction,"[""Identify the countries, their capitals, and ...","[""Identify the countries, their captals, and t...",...,,,,,,,,,,
7,MTBench_003,MTBench_003_rate_0.4,MTBench,0.4,raw_0.4.jsonl,55,135,extraction,"[""Identify the countries, their capitals, and ...","[""Identify th coumtries, thei capitals, and th...",...,,,,,,,,,,
8,MTBench_003,MTBench_003_rate_0.7,MTBench,0.7,raw_0.7.jsonl,55,135,extraction,"[""Identify the countries, their capitals, and ...","[""Identigy the contries, theur caputals, and t...",...,,,,,,,,,,
9,MTBench_004,MTBench_004_rate_0.1,MTBench,0.1,raw_0.1.jsonl,44,124,coding,"[""Here is a Python function to find the length...","[""Hee is a Python function to find the length ...",...,,,,,,,,,,



Column check:
Empty Original_Prompt rows: 0
Empty Modified_Prompt rows: 0
Empty Gold_Answer rows: 6
Empty Question_ID rows: 0
Empty Category rows: 0

Rows per typo-rate stratum:
Typo_Rate
0.1    5
0.4    5
0.7    5
Name: count, dtype: int64

Matched sampling check passed: every Base_ID has 3 typo-rate versions.
Rate-label check passed: every Base_ID contains 0.1, 0.4, and 0.7.

Note: Empty Gold_Answer values may be normal for MT-Bench because it is commonly evaluated using an LLM judge rather than one fixed answer.

CSV saved successfully: /content/MTBench_matched_stratified_sample_n5_seed42.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>